In [1]:
import json
from datetime import datetime

In [2]:
RAW_JSON_DATA = """
[
    {"id": "101", "name": "alice smith", "email": "ALICE@example.com", "signup_date": "2026-01-15", "status": "active", "tier": "premium", "spent": 150.50},
    {"id": "102", "name": "bob jones", "email": "bob@example.com", "signup_date": "2026-02-20", "status": "pending", "tier": "free", "spent": 0.00},
    {"id": "103", "name": "charlie brown", "email": "charlie@domain.com", "signup_date": "2026-01-10", "status": "active", "tier": "premium", "spent": 99.99},
    {"id": "104", "name": "deleted user", "email": "none@domain.com", "signup_date": "2026-03-01", "status": "inactive", "tier": "free", "spent": 12.00},
    {"id": "105", "name": "diana prince", "email": "diana@example.com", "signup_date": "2026-02-14", "status": "active", "tier": "standard", "spent": 45.00},
    {"id": "106", "name": "evan wright", "email": "evan@domain.com", "signup_date": "2026-03-05", "status": "active", "tier": "standard", "spent": 60.00}
]
"""

In [11]:
def extract(json_string):
    return json.loads(json_string)

In [30]:
def transform(raw_records):
    curr_time = datetime.now()
    cleaned_records = []

    for record in raw_records:
        if record.get("status","").lower() == "inactive":
            continue

        try:
            cleaned_records.append({
                "user_id": int(record.get("id")),
                "full_name": record.get("name").title(),
                "email_address": record.get("email").lower(),
                "signup_date": record.get("signup_date"),
                "account_status": record.get("status").upper(),
                "tier": record.get("tier").lower(),
                "total_spent": float(record.get("spent", 0.0))
            })
        except Exception as e:
            print(f"Exception occurred: {e}")
            continue

    tier_metrics = {}
    for r in cleaned_records:
        tier = r.get("tier")
        spent = r.get("total_spent")

        if tier not in tier_metrics:
            tier_metrics[tier] = {"user_count": 0, "total_revenue": 0.0}

        tier_metrics[tier]["user_count"] += 1
        tier_metrics[tier]["total_revenue"] += spent

    print(tier_metrics.items())
    for tier, stats in tier_metrics.items():
        stats["avg_spend_per_user"] = round(stats["total_revenue"] / stats["user_count"], 2)
        stats["total_revenue"] = round(stats["total_revenue"], 2)

    sorted_records = sorted(
        cleaned_records,
        key=lambda x: (-x["total_spent"], x["user_id"])
    )

    print("Transform Succeeded")

    return {
        "metadata": {
            "generated_at": datetime.strftime(curr_time, '%Y-%m-%dT%H:%M:%S'),
            "total_active_records": len(sorted_records)
        },
        "aggregated_metrics_by_tier": tier_metrics,
        "records": sorted_records
    }
    

In [31]:
def load(payload, output_filepath):
    with open(output_filepath, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=4)
    print("Load Succeeded")

In [32]:
def run_pipeline():
    raw_data = extract(RAW_JSON_DATA)
    final_payload = transform(raw_data)
    load(final_payload, "../output/etl_analytics_output.json")

In [33]:
run_pipeline()

dict_items([('premium', {'user_count': 2, 'total_revenue': 250.49}), ('free', {'user_count': 1, 'total_revenue': 0.0}), ('standard', {'user_count': 2, 'total_revenue': 105.0})])
Transform Succeeded
Load Succeeded
